# Preparação — quadros e calibração da câmera

Execute este notebook uma vez para cada captura antes de usar `reconstruction.ipynb`. Aqui ficam as tarefas que precisam de intervenção: extração/verificação dos quadros e seleção manual das interseções do papel de 5 mm para estimar `K`. A pipeline principal só carrega o arquivo salvo e roda sem cliques.

Para a captura `pedra_cubica_base`, os 80 quadros já existentes são reutilizados. Se alterar a quantidade, trocar o vídeo ou criar outra captura, atualize os parâmetros abaixo e execute novamente esta preparação. A calibração é salva em `outputs/<objeto>/<captura>_<N>quadros_calibracao/` e reutilizada pela pipeline até os nomes dos quadros ou parâmetros mudarem.


In [9]:
from pathlib import Path
import json
import numpy as np
import cv2

ROOT = Path.cwd()
if not (ROOT / "extrair_imagens.py").is_file() and (ROOT / "t2" / "extrair_imagens.py").is_file():
    ROOT = ROOT / "t2"

# Edite estes valores para preparar outra captura.
CAPTURE_ID = "pedra_cubica_base"
OBJECT_NAME = "pedra_cubica"
VIDEO_PATH = ROOT / "images" / "pedra_cubica" / "P1011579.MOV"
FRAMES_DIR = ROOT / "images" / "pedra_cubica"
NUM_FRAMES = 80
REEXTRACT_FRAMES = False
FRAME_INDICES = None  # None escolhe automaticamente os sete quadros finais.
GRID_SIZE = (5, 5)  # interseções; a região clicada abrange 4 x 4 quadrados
SQUARE_SIZE_M = 0.005
FORCE_RECALIBRATION = False

from extrair_imagens import extrair_imagens

FRAMES_DIR.mkdir(parents=True, exist_ok=True)
frame_paths = sorted(FRAMES_DIR.glob(f"quadro_*_de_{NUM_FRAMES:04d}_*.jpg"))
if NUM_FRAMES == 80 and len(frame_paths) != NUM_FRAMES:
    frame_paths = sorted(FRAMES_DIR.glob("frame_*.jpg"))
if REEXTRACT_FRAMES or len(frame_paths) != NUM_FRAMES:
    if not VIDEO_PATH.is_file():
        raise FileNotFoundError(f"Vídeo não encontrado: {VIDEO_PATH}")
    frame_paths = extrair_imagens(VIDEO_PATH, FRAMES_DIR, NUM_FRAMES)
if len(frame_paths) < 5:
    raise RuntimeError("A calibração requer pelo menos cinco quadros.")
if FRAME_INDICES is None:
    FRAME_INDICES = list(range(max(0, len(frame_paths) - 7), len(frame_paths)))
if len(FRAME_INDICES) < 5 or any(i < 0 or i >= len(frame_paths) for i in FRAME_INDICES):
    raise ValueError("FRAME_INDICES deve conter pelo menos cinco índices válidos.")
calibration_paths = [frame_paths[i] for i in FRAME_INDICES]
print(f"{len(frame_paths)} quadros prontos em {FRAMES_DIR}")
print("Quadros escolhidos para calibração:")
for path in calibration_paths:
    print(" -", path.name)


80 quadros prontos em C:\01b_TUM\26_WiSe_Forschung\CompVision\t2\images\pedra_cubica
Quadros escolhidos para calibração:
 - frame_0074_t00087.80s.jpg
 - frame_0075_t00089.00s.jpg
 - frame_0076_t00090.20s.jpg
 - frame_0077_t00091.40s.jpg
 - frame_0078_t00092.60s.jpg
 - frame_0079_t00093.80s.jpg
 - frame_0080_t00095.02s.jpg


## Calibração manual (única etapa com cliques)

Nos sete quadros finais selecionados, clique quatro interseções que delimitam a mesma região plana de 5 × 5 interseções (4 × 4 quadrados de 5 mm). Use a ordem: superior esquerda, superior direita, inferior direita, inferior esquerda. As posições escolhidas podem ser diferentes entre quadros; use apenas a mesma contagem e ordem. Depois de salvar o arquivo de calibração, a pipeline principal não pedirá esses cliques novamente.


In [10]:
from calibracao_grade import calibrar_camera_papel_quadriculado

output_dir = ROOT / "outputs" / OBJECT_NAME / f"{CAPTURE_ID}_{len(frame_paths)}quadros_calibracao"
output_dir.mkdir(parents=True, exist_ok=True)
frame_tag = f"{len(frame_paths)}quadros"
cache_path = output_dir / f"calibracao_camera_{frame_tag}.npz"
metadata_path = output_dir / f"calibracao_camera_{frame_tag}.json"
settings = {"quadros": [p.name for p in calibration_paths], "grid_size": list(GRID_SIZE), "square_size_m": SQUARE_SIZE_M}

cached = False
if cache_path.is_file() and metadata_path.is_file() and not FORCE_RECALIBRATION:
    old_settings = json.loads(metadata_path.read_text(encoding="utf-8")).get("settings")
    if old_settings == settings:
        with np.load(cache_path) as stored:
            result = {"camera_matrix": stored["camera_matrix"], "dist_coeffs": stored["dist_coeffs"],
                      "rms_px": float(stored["rms_px"]), "per_view_rms_px": stored["per_view_rms_px"].tolist(),
                      "image_size": tuple(stored["image_size"]), "image_points": [row for row in stored["image_points"]],
                      "image_paths": calibration_paths, "overlays": []}
        cached = True

if not cached:
    result = calibrar_camera_papel_quadriculado(calibration_paths, grid_size=GRID_SIZE, square_size_m=SQUARE_SIZE_M)
    np.savez_compressed(cache_path, camera_matrix=result["camera_matrix"], dist_coeffs=result["dist_coeffs"],
                        rms_px=result["rms_px"], per_view_rms_px=result["per_view_rms_px"],
                        image_size=result["image_size"], image_points=np.stack(result["image_points"]))
    metadata_path.write_text(json.dumps({"settings": settings, "rms_px": result["rms_px"],
                            "per_view_rms_px": result["per_view_rms_px"]}, indent=2), encoding="utf-8")
    for path, overlay in zip(calibration_paths, result["overlays"]):
        cv2.imwrite(str(output_dir / f"{path.stem}_sobreposicao_grade.jpg"), overlay)

print(f"Calibração salva: {cache_path}")
print(f"RMS de reprojeção: {result['rms_px']:.3f} px")
print("Matriz K =\n", result["camera_matrix"])


Calibração salva: C:\01b_TUM\26_WiSe_Forschung\CompVision\t2\outputs\pedra_cubica\pedra_cubica_base_80quadros_calibracao\calibracao_camera_80quadros.npz
RMS de reprojeção: 1.281 px
Matriz K =
 [[1.50490330e+03 0.00000000e+00 9.60039819e+02]
 [0.00000000e+00 1.20851341e+03 5.39332233e+02]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]
